# Translate JLPT Vocabulary to Indonesian

Notebook ini membaca file `Data/Vocabulary/n*_vocab_cleaned.csv`, memanggil endpoint lokal `http://localhost:11434/v1/completions`, lalu membuat output CSV baru dengan kolom `Indonesian`.

Output tidak menimpa file asli. File hasil akan masuk ke `Data/Vocabulary/Indonesian/`.

In [2]:
from pathlib import Path
import csv
import time

import requests


# Kalau notebook dibuka dari folder Preprocessing, naik satu folder ke project root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Data" / "Vocabulary").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "Data" / "Vocabulary"
OUTPUT_DIR = INPUT_DIR / "Indonesian"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINT = "http://localhost:11434/v1/completions"
MODEL = "translategemma:4b"

TEXT_COLUMN = "Kanji"
READING_COLUMN = "Reading"
OUTPUT_COLUMN = "Indonesian"

PROMPT_TEMPLATE = """Translate the following text from Japanese (ja) to Indonesian (id). Output only the translation.

Text: {kanji}
Translation:"""

# Set angka kecil, misalnya 20, untuk test dulu. Pakai None untuk semua baris.
LIMIT_ROWS = None

# Simpan progres setiap N terjemahan supaya aman kalau proses terhenti.
SAVE_EVERY = 10
REQUEST_DELAY_SECONDS = 0.05
MAX_RETRIES = 3
TIMEOUT_SECONDS = 60

SOURCE_FILES = sorted(INPUT_DIR.glob("n*_vocab_cleaned.csv"))
SOURCE_FILES


[WindowsPath('d:/Codingan Pribadi/project pribadi/Kanji App For JLPT/Data/Vocabulary/n1_vocab_cleaned.csv'),
 WindowsPath('d:/Codingan Pribadi/project pribadi/Kanji App For JLPT/Data/Vocabulary/n2_vocab_cleaned.csv'),
 WindowsPath('d:/Codingan Pribadi/project pribadi/Kanji App For JLPT/Data/Vocabulary/n3_vocab_cleaned.csv'),
 WindowsPath('d:/Codingan Pribadi/project pribadi/Kanji App For JLPT/Data/Vocabulary/n4_vocab_cleaned.csv'),
 WindowsPath('d:/Codingan Pribadi/project pribadi/Kanji App For JLPT/Data/Vocabulary/n5_vocab_cleaned.csv')]

In [3]:
def read_csv_rows(path: Path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        return list(reader), list(reader.fieldnames or [])


def write_csv_rows(path: Path, rows, fieldnames):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def clean_translation(text: str) -> str:
    text = str(text or "").strip()

    for prefix in ("Translation:", "Terjemahan:", "Indonesian:", "Output:"):
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    return text.strip().strip('"').strip("'")


def translate_to_indonesian(kanji: str) -> str:
    payload = {
        "model": MODEL,
        "prompt": PROMPT_TEMPLATE.format(kanji=kanji),
        "max_tokens": 128,
        "temperature": 0.0,
        "top_p": 1.0,
        "stream": False,
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(ENDPOINT, json=payload, timeout=TIMEOUT_SECONDS)
            response.raise_for_status()
            data = response.json()
            text = data["choices"][0].get("text", "")
            return clean_translation(text)
        except Exception as error:
            last_error = error
            sleep_seconds = min(2 ** attempt, 10)
            print(f"Retry {attempt}/{MAX_RETRIES} for {kanji!r}: {error}")
            time.sleep(sleep_seconds)

    raise RuntimeError(f"Failed to translate {kanji!r}") from last_error


def merge_existing_translations(output_path: Path, rows):
    if not output_path.exists():
        return rows

    existing_rows, _ = read_csv_rows(output_path)
    existing_by_key = {
        (row.get(TEXT_COLUMN, ""), row.get(READING_COLUMN, "")): row.get(OUTPUT_COLUMN, "")
        for row in existing_rows
        if row.get(OUTPUT_COLUMN)
    }

    for row in rows:
        key = (row.get(TEXT_COLUMN, ""), row.get(READING_COLUMN, ""))
        if not row.get(OUTPUT_COLUMN) and key in existing_by_key:
            row[OUTPUT_COLUMN] = existing_by_key[key]

    return rows


In [3]:
# Test satu request dulu sebelum menjalankan seluruh dataset.
# Pastikan Ollama/server translategemma sudah aktif di http://localhost:11434.
translate_to_indonesian("学校")


'Sekolah'

In [4]:
def translate_file(source_path: Path):
    output_path = OUTPUT_DIR / source_path.name
    rows, input_fieldnames = read_csv_rows(source_path)
    fieldnames = [*input_fieldnames]

    if OUTPUT_COLUMN not in fieldnames:
        fieldnames.append(OUTPUT_COLUMN)

    for row in rows:
        row.setdefault(OUTPUT_COLUMN, "")

    rows = merge_existing_translations(output_path, rows)
    pending_indexes = [
        index
        for index, row in enumerate(rows)
        if row.get(TEXT_COLUMN) and not row.get(OUTPUT_COLUMN)
    ]

    if LIMIT_ROWS is not None:
        pending_indexes = pending_indexes[:LIMIT_ROWS]

    print(f"\n{source_path.name}: {len(pending_indexes)} rows to translate")

    translated_count = 0
    for number, row_index in enumerate(pending_indexes, start=1):
        row = rows[row_index]
        kanji = row[TEXT_COLUMN]
        row[OUTPUT_COLUMN] = translate_to_indonesian(kanji)
        translated_count += 1

        print(f"[{source_path.name}] {number}/{len(pending_indexes)} {kanji} -> {row[OUTPUT_COLUMN]}")

        if translated_count % SAVE_EVERY == 0:
            write_csv_rows(output_path, rows, fieldnames)

        time.sleep(REQUEST_DELAY_SECONDS)

    write_csv_rows(output_path, rows, fieldnames)
    print(f"Saved: {output_path}")


for source_file in SOURCE_FILES:
    translate_file(source_file)



n1_vocab_cleaned.csv: 3475 rows to translate
[n1_vocab_cleaned.csv] 1/3475 嗚呼 -> Ah, sial
[n1_vocab_cleaned.csv] 2/3475 相 -> Hubungan
[n1_vocab_cleaned.csv] 3/3475 相変わらず -> Seperti biasa
[n1_vocab_cleaned.csv] 4/3475 愛想 -> Kesopanan
[n1_vocab_cleaned.csv] 5/3475 相対 -> Relatif
[n1_vocab_cleaned.csv] 6/3475 間柄 -> Hubungan
[n1_vocab_cleaned.csv] 7/3475 愛憎 -> Cinta dan benci
[n1_vocab_cleaned.csv] 8/3475 合間 -> Di antara waktu
[n1_vocab_cleaned.csv] 9/3475 曖昧 -> Tidak jelas
[n1_vocab_cleaned.csv] 10/3475 敢えて -> Dengan sengaja
[n1_vocab_cleaned.csv] 11/3475 仰ぐ -> Menatap
[n1_vocab_cleaned.csv] 12/3475 垢 -> Akun
[n1_vocab_cleaned.csv] 13/3475 亜科 -> Subfamili
[n1_vocab_cleaned.csv] 14/3475 銅 -> Tembaga
[n1_vocab_cleaned.csv] 15/3475 証 -> Bukti
[n1_vocab_cleaned.csv] 16/3475 赤字 -> Merugi
[n1_vocab_cleaned.csv] 17/3475 明かす -> Menjelaskan
[n1_vocab_cleaned.csv] 18/3475 赤ちゃん -> Bayi
[n1_vocab_cleaned.csv] 19/3475 明白 -> Jelas
[n1_vocab_cleaned.csv] 20/3475 赤らむ -> Merah
[n1_vocab_cleaned.csv] 21/34

In [5]:
# Japanese -> English vocabulary translation.
# Jalankan cell setup + helper di atas dulu, lalu jalankan cell ini.

ENGLISH_OUTPUT_DIR = INPUT_DIR / "English"
ENGLISH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENGLISH_OUTPUT_COLUMN = "English"
ENGLISH_LIMIT_ROWS = None  # Ganti ke angka kecil, misalnya 20, untuk test dulu.

ENGLISH_PROMPT_TEMPLATE = """Translate the following vocabulary from Japanese (ja) to English (en). Output only the translation.

Vocabulary: {kanji}
Translation:"""


def translate_to_english(kanji: str) -> str:
    payload = {
        "model": MODEL,
        "prompt": ENGLISH_PROMPT_TEMPLATE.format(kanji=kanji),
        "max_tokens": 128,
        "temperature": 0.0,
        "top_p": 1.0,
        "stream": False,
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(ENDPOINT, json=payload, timeout=TIMEOUT_SECONDS)
            response.raise_for_status()
            data = response.json()
            text = data["choices"][0].get("text", "")
            return clean_translation(text)
        except Exception as error:
            last_error = error
            sleep_seconds = min(2 ** attempt, 10)
            print(f"Retry {attempt}/{MAX_RETRIES} for {kanji!r}: {error}")
            time.sleep(sleep_seconds)

    raise RuntimeError(f"Failed to translate {kanji!r}") from last_error


def merge_existing_english_translations(output_path: Path, rows):
    if not output_path.exists():
        return rows

    existing_rows, _ = read_csv_rows(output_path)
    existing_by_key = {
        (row.get(TEXT_COLUMN, ""), row.get(READING_COLUMN, "")): row.get(ENGLISH_OUTPUT_COLUMN, "")
        for row in existing_rows
        if row.get(ENGLISH_OUTPUT_COLUMN)
    }

    for row in rows:
        key = (row.get(TEXT_COLUMN, ""), row.get(READING_COLUMN, ""))
        if not row.get(ENGLISH_OUTPUT_COLUMN) and key in existing_by_key:
            row[ENGLISH_OUTPUT_COLUMN] = existing_by_key[key]

    return rows


def translate_file_to_english(source_path: Path):
    output_path = ENGLISH_OUTPUT_DIR / source_path.name
    rows, input_fieldnames = read_csv_rows(source_path)
    fieldnames = [*input_fieldnames]

    if ENGLISH_OUTPUT_COLUMN not in fieldnames:
        fieldnames.append(ENGLISH_OUTPUT_COLUMN)

    for row in rows:
        row.setdefault(ENGLISH_OUTPUT_COLUMN, "")

    rows = merge_existing_english_translations(output_path, rows)
    pending_indexes = [
        index
        for index, row in enumerate(rows)
        if row.get(TEXT_COLUMN) and not row.get(ENGLISH_OUTPUT_COLUMN)
    ]

    if ENGLISH_LIMIT_ROWS is not None:
        pending_indexes = pending_indexes[:ENGLISH_LIMIT_ROWS]

    print(f"\n{source_path.name}: {len(pending_indexes)} English rows to translate")

    translated_count = 0
    for number, row_index in enumerate(pending_indexes, start=1):
        row = rows[row_index]
        kanji = row[TEXT_COLUMN]
        row[ENGLISH_OUTPUT_COLUMN] = translate_to_english(kanji)
        translated_count += 1

        print(f"[{source_path.name}] {number}/{len(pending_indexes)} {kanji} -> {row[ENGLISH_OUTPUT_COLUMN]}")

        if translated_count % SAVE_EVERY == 0:
            write_csv_rows(output_path, rows, fieldnames)

        time.sleep(REQUEST_DELAY_SECONDS)

    write_csv_rows(output_path, rows, fieldnames)
    print(f"Saved: {output_path}")


for source_file in SOURCE_FILES:
    translate_file_to_english(source_file)



n1_vocab_cleaned.csv: 3475 English rows to translate
[n1_vocab_cleaned.csv] 1/3475 嗚呼 -> Alas
[n1_vocab_cleaned.csv] 2/3475 相 -> relative
[n1_vocab_cleaned.csv] 3/3475 相変わらず -> as usual
[n1_vocab_cleaned.csv] 4/3475 愛想 -> affection, fondness, amiable
[n1_vocab_cleaned.csv] 5/3475 相対 -> relative
[n1_vocab_cleaned.csv] 6/3475 間柄 -> relationship
[n1_vocab_cleaned.csv] 7/3475 愛憎 -> love and hate
[n1_vocab_cleaned.csv] 8/3475 合間 -> intervals
[n1_vocab_cleaned.csv] 9/3475 曖昧 -> vague
[n1_vocab_cleaned.csv] 10/3475 敢えて -> deliberately; intentionally; boldly; daringly
[n1_vocab_cleaned.csv] 11/3475 仰ぐ -> to gaze at; to look up at
[n1_vocab_cleaned.csv] 12/3475 垢 -> skin, dirt, grime, filth, appearance
[n1_vocab_cleaned.csv] 13/3475 亜科 -> Subfamily
[n1_vocab_cleaned.csv] 14/3475 銅 -> copper
[n1_vocab_cleaned.csv] 15/3475 証 -> evidence
[n1_vocab_cleaned.csv] 16/3475 赤字 -> red ink
[n1_vocab_cleaned.csv] 17/3475 明かす -> to reveal, to clarify, to explain
[n1_vocab_cleaned.csv] 18/3475 赤ちゃん -> baby
